In [1]:
!git clone https://github.com/wenerd569/Speech-recognition.git
!curl http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz --output speech_commands_v0.02.tar.gz


Cloning into 'Speech-recognition'...
remote: Enumerating objects: 196, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 196 (delta 0), reused 0 (delta 0), pack-reused 190 (from 1)
Receiving objects: 100% (196/196), 38.95 KiB | 12.98 MiB/s, done.
Resolving deltas: 100% (104/104), done.
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2316M  100 2316M    0     0  22.8M      0  0:01:41  0:01:41 --:--:-- 23.6M


In [2]:

!mkdir -p ./Speech-recognition/data/speech_commands_v0.02
!tar -xzf speech_commands_v0.02.tar.gz -C Speech-recognition/data/speech_commands_v0.02



In [ ]:
%cd Speech-recognition
!apt install -y ffmpeg
!pwd
!git fetch
!git pull
!pip install -r requirements.txt
!pip install --force-reinstall --no-cache-dir numpy scipy



[Errno 2] No such file or directory: 'Speech-recognition'
/content/Speech-recognition
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:6.1.1-3ubuntu5).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
/content/Speech-recognition
From https://github.com/wenerd569/Speech-recognition
 * [new branch]      wenerd4    -> origin/wenerd4
Already up to date.
  Using cached pandas-3.0.6-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached torch-2.14.1-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (37 kB)
  Using cached torchcodec-0.16.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.7 kB)
  Using cached torchinfo-1.8.0-py3-none-any.whl.metadata (21 kB)
  Using cached torchmetrics-1.9.0-py3-none-any.whl.metadata (23 kB)
ERROR: Could not find a version that satisfies the requirement train_evaluate==0.4.6 (from versions: none)
ERROR: 

In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader, DataLoader

from src.train_evaluate import train

from src.data_preparing import MelSpecDataset, init_datasets
from src.base_net import BaseNet
from src.mean_net import MeanNet
from src.union_net import UnionNet

from src.wav_preproccess import transform_wavs_to_tensors


ModuleNotFoundError: No module named 'torchmetrics'

In [7]:
DIR_CURRENT = os.path.dirname(os.path.abspath(__file__))
DIR_TO_UPLOAD_FROM = os.path.join(os.path.dirname(DIR_CURRENT), "data/speech_commands_v0.02")
DIR_TO_SAVE_TO = os.path.join(os.path.dirname(DIR_CURRENT), "data/data_proccessed")
DF_PATH = os.path.join(DIR_TO_SAVE_TO, "DF.csv")



Sample_rate = 16000
N_fft = 1024
N_mels = 128
Hop_length = 128
Target_T = Sample_rate // Hop_length + 1

transform_wavs_to_tensors(DIR_TO_UPLOAD_FROM, DIR_TO_SAVE_TO, Sample_rate, N_fft, N_mels, Hop_length)
init_datasets(DIR_TO_UPLOAD_FROM, DIR_TO_SAVE_TO)

NameError: name '__file__' is not defined

In [6]:

BATCH_SIZE = 32
LABELS = sorted(pd.read_csv(DF_PATH)["label"].unique())
LABEL_TO_INDEX = {label: index for index, label in enumerate(LABELS)}
NUM_CLASSES = len(LABELS)

BASE_NET_INPUT_SHAPE = (32, 1, N_mels, Target_T)


def get_loader(df_csv_name: str):
    df_train = pd.read_csv(os.path.join(DIR_TO_SAVE_TO, df_csv_name))
    dataset = MelSpecgDataset(df_train, DIR_TO_SAVE_TO, LABEL_TO_INDEX)
    train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    return train_loader



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

base_cnn = BaseNet(kernel_sizes=[3, 3, 3, 3], strides=[1, 1, 1, 1], paddings=[1, 1, 1, 1])

BASE_NET_OUTPUT_SHAPE = summary(base_cnn.features, input_size = BASE_NET_INPUT_SHAPE, verbose=0).summary_list[-1].output_size
   
mean_net = MeanNet(output_features=len(LABEL_TO_INDEX), input_features=BASE_NET_OUTPUT_SHAPE[1]*BASE_NET_OUTPUT_SHAPE[2])

net = UnionNet(
    base_cnn,
    mean_net,
)

optimizer = Adam(net.parameters(), lr=0.001     )
loss = nn.NLLLoss()

train_loader = get_loader("df_train.csv")
validation_loader = get_loader("df_validation.csv")

train(net, epochs=40, train_loader=train_loader, validate_loader=validation_loader, optimizer=optimizer, loss=loss, num_classes=NUM_CLASSES, device=device)


NameError: name 'DF_PATH' is not defined